In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)

anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
if anthropic_api_key:
    print(f"OpenAI API Key exists and begins {anthropic_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# MODEL = 'llama3.2'
MODEL = 'claude-sonnet-4-5-20250929'

anthropic_url = "https://api.anthropic.com/v1/"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)

In [ ]:
car_prices = {"harrier": "1000 Rs", "thar": "500 Rs", "fortuner": "1000 Rs", "altroz": "900 Rs", "swift":"100 Rs"}

def get_car_price(car):
    print(f"Tool called for {car}")
    price = car_prices.get(car.lower(), "Unknown ticket price")
    return f"The price of {car} is {price}"

In [ ]:
price_function = {
    "name": "get_car_price",
    "description": "Get the price a car",
    "parameters": {
        "type": "object",
        "properties": {
            "car_model": {
                "type": "string",
                "description": "Car model name",
            },
        },
        "required": ["car_model"],
        "additionalProperties": False
    }
}

tools = [{"type": "function", "function": price_function}]

def handle_tool_call(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_car_price":
            arguments = json.loads(tool_call.function.arguments)
            car = arguments.get('car_model')
            price_details = get_car_price(car)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
system_message = "You are a helpful assistant at a car showroom. Give short, courteous answers, no more than 1 sentence. \
Always be accurate. If you don't know the answer, say so. "

def chat(user_message, history):
    cleaned_history =  [{'role': h['role'], 'content':h['content']} for h in history]
    messages = [{'role': 'system', 'content': system_message }] + cleaned_history + [{'role': 'user', 'content': user_message }]
    response = anthropic.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    while response.choices[0].finish_reason=="tool_calls":
        bot_message = response.choices[0].message
        tool_response = handle_tool_call(bot_message)
        messages.append(bot_message)
        messages.extend(tool_response)
        response = anthropic.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()